# Exploring the tree of a Tree SAE

Loads a released Tree SAE, collects its activations on Pile validation text, and walks the learned parent/child structure (Section 6.2).

In [ ]:
import torch
from transformer_lens import HookedTransformer

from tree_sae.analysis.activations import llm_activations, load_eval_tokens, sae_sparse_activations
from tree_sae.models.io import from_pretrained

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device, dtype=torch.bfloat16)

In [ ]:
sae = from_pretrained("gpt2-small/tree_sae_2layer_l0_48", device=device).to(torch.bfloat16)
n_seqs, seq_len = 2000, 128  # the paper uses 10,000 x 128 tokens
tokens = load_eval_tokens(n_seqs, seq_len, "gpt2")
acts = llm_activations(model, tokens, sae.cfg.hook_name)
indices, values = sae_sparse_activations(sae, acts, 128, device, verbose=True)
indices[values < 1e-3] = sae.cfg.d_sae  # mark inactive entries

## The tree structure

`sae.parent_index(l)` is the allocation vector $a_l$ of privilege layer $l$; the root is encoded as `sae.root_index(l)`.

In [ ]:
from tree_sae.analysis.cooccurrence import children_by_parent

for layer in range(1, sae.n_layers):
    families = children_by_parent(sae, layer)
    n_root = (sae.parent_index(layer) == sae.root_index(layer)).sum().item()
    print(f"layer {layer}: {sae.layer_size(layer)} features, {len(families)} parents, {n_root} attached to the root")

root = 3410
print("children of", root, "->", sae.children_of(root))

## What do a parent and its children fire on?

In [ ]:
from tree_sae.analysis.features import firing_rate, max_activating_examples, print_examples

shape = (n_seqs, seq_len, -1)
for feature in [root, *sae.children_of(root)[:3]]:
    print(
        f"feature {feature}: fires on {100 * firing_rate(feature, indices.view(shape), values.view(shape)):.3f}% of tokens"
    )
    print_examples(max_activating_examples(model, feature, indices.view(shape), values.view(shape), tokens, k=5))

## Interactive tree (Figs. 17-20)

In [ ]:
from tree_sae.analysis.tree_vis import build_feature_data, save_feature_tree, subtree

tree = subtree(sae, root)
data = build_feature_data(tree, model, indices.view(shape), values.view(shape), tokens, k=10)
save_feature_tree(tree, data, f"feature_tree_{root}.html")

## Sibling co-occurrence (Table 3)

In [ ]:
from tree_sae.analysis.cooccurrence import average_sibling_cooccurrence

average_sibling_cooccurrence(sae, indices)